# IMPORT LIBRARY

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# CONNECTING TO DATABASE

In [ ]:
load_dotenv("../.env")

conn = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connection created")

# PROPERTY PRICE DRIVERS & PREDICTIVE MODELING

##
RETRIVE DATA

In [ ]:
dataset = '''SELECT
                t.property_id
                ,t.deal_status
                ,t.deal_price
                ,t.deal_date
                ,t.payment_mode
                ,t.city
                ,l.listed_price
                ,l.days_on_market
                ,p.property_type
                ,p.size_sqm
                ,p.has_parking
                ,p.near_transit
                ,p.near_school
                ,p.year_built
            FROM transactions t
            LEFT JOIN listings l ON t.listing_id = l.listing_id
            LEFT JOIN properties p ON t.property_id = p.property_id
            WHERE DATE_PART('year',deal_date)<=2024
            '''
df_dataset = pd.read_sql(dataset,conn)
len(df_dataset)

##
LOAD DATA AND DATA PREPARATION

In [ ]:
df_completed = df_dataset[df_dataset['deal_status']=='Completed']

print(df_completed.isna().sum())
print(f'\nNumber of Duplicate : {df_completed.duplicated().sum()}')

bool_cols = ['has_parking', 'near_transit', 'near_school']
for col in bool_cols:
    df_completed[col] = df_completed[col].astype(str).str.strip().str.lower()=='true'

##
FEATURE ENGINEERING

In [ ]:
df_completed['deal_date'] =(\
    pd.to_datetime(df_completed['deal_date'])
)

df_completed['year_deal'] = (\
    df_completed['deal_date']
    .dt.year
)

df_completed['month_deal'] = (\
    df_completed['deal_date']
    .dt.month
)

df_completed['property_age'] = (\
    (df_completed['deal_date'].dt.year)-
    df_completed['year_built']
)

##
CHECKING OUTLIER

In [ ]:
columns = ['deal_price','listed_price','size_sqm'
,'days_on_market','property_age']

for col in columns:
    Q1 = df_completed[col].quantile(0.25)
    Q3 = df_completed[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers =(\
        df_completed[(df_completed[col] < lower_bound) | 
        (df_completed[col] > upper_bound)]
    )
    print(f"Outlier pada '{col}': {len(outliers)} baris ({len(outliers)/len(df_completed)*100:.1f}%)")

##
CORRELATION TEST

In [ ]:
corr_matrix = (\
    df_completed[['deal_price','listed_price','size_sqm'
                ,'days_on_market','property_age']].corr()
)

print(corr_matrix['deal_price'].sort_values(ascending=False).to_string())
plt.figure(figsize=(8, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Heatmap Korelasi Fitur Numerik vs Deal Price', fontweight='bold')
plt.tight_layout()
plt.show()

##
ENCODING

##
FEATURE SELECTION

In [ ]:
df_completed.columns

In [ ]:
features_cols = ['payment_mode','city', 'days_on_market', 
                'property_type', 'size_sqm','has_parking', 'near_transit',
                'near_school','property_age','year_deal','month_deal']

target_col = 'deal_price'

x = df_completed[features_cols].copy()
y = df_completed[target_col].copy()

x.info()

In [ ]:
cat_cols = ['payment_mode', 'city', 'property_type','month_deal','year_deal']

x_encoded = pd.get_dummies(x, columns = cat_cols,drop_first = True)

x_encoded

##
SPLIT TRAIN-TEST

In [ ]:
x_train, x_test, y_train, y_test =(
    train_test_split(
    x_encoded, 
    y, 
    test_size = 0.2, 
    random_state = 42
    )
)

print(f"\nData Train: {x_train.shape[0]} baris | Data Test: {x_test.shape[0]} baris")

##
IDENTIFYING BASELINE CATEGORIES

In [ ]:

encoded_columns = ['payment_mode', 'city', 'property_type', 'month_deal', 'year_deal']


print("Baseline for Each Categories")


for col in encoded_columns:
    all_categories = set(df_completed[col].astype(str).unique())
    prefix = f"{col}_"
    categories_in_model = set([
        str(feature).replace(prefix, '') 
        for feature in x_train.columns 
        if str(feature).startswith(prefix)
    ])
    baseline = all_categories - categories_in_model

    baseline_value = list(baseline)[0] if baseline else "None (All categories are present)"
    print(f"• Baseline for '{col}' : {baseline_value}")



# MODELLING

## LINEAR REGRESSION

In [ ]:

x_train_lr = x_train.copy()
x_test_lr = x_test.copy()


y_train_lr = np.log1p(y_train)
y_test_lr = np.log1p(y_test)


x_train_lr['size_sqm'] = np.log1p(x_train_lr['size_sqm'])
x_test_lr['size_sqm'] = np.log1p(x_test_lr['size_sqm'])


num_cols = ['days_on_market', 'size_sqm', 'property_age']


x_train_lr_scaled = x_train_lr.copy()
x_test_lr_scaled = x_test_lr.copy()

scaler = StandardScaler()
x_train_lr_scaled[num_cols] = scaler.fit_transform(x_train_lr[num_cols])
x_test_lr_scaled[num_cols] = scaler.transform(x_test_lr[num_cols])


lr_model = LinearRegression()
lr_model.fit(x_train_lr_scaled, y_train_lr)

#Prediction
y_pred_lr_log = lr_model.predict(x_test_lr_scaled)


y_pred_lr = np.expm1(y_pred_lr_log)
y_test_actual_lr = np.expm1(y_test_lr)


mae_lr = mean_absolute_error(y_test_actual_lr, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test_actual_lr, y_pred_lr))
r2_lr = r2_score(y_test_actual_lr, y_pred_lr)

print(f"\n--- Linear Regression Result---")
print(f"MAE  : BDT {mae_lr:,.0f}")
print(f"RMSE : BDT {rmse_lr:,.0f}")
print(f"R2   : {r2_lr:.4f} ({r2_lr*100:.2f}%)")

In [ ]:
feature_names = x_train_lr.columns

coef_lr = lr_model.coef_


df_coef = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient Value (b)': coef_lr
}).sort_values(by='Coefficient Value (b)', key=abs, ascending=False).reset_index(drop = True)


print(f"Intercept Values (Konstanta a) : {lr_model.intercept_:.4f}\n")

print("--- Coefficient Fitures ---")
df_coef

## RANDOM FOREST

In [ ]:
x_train_rf = x_train.copy()
x_test_rf  = x_test.copy()

y_train_rf = np.log1p(y_train)
y_test_rf  = np.log1p(y_test)


rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(x_train_rf, y_train_rf)


y_pred_rf_log = rf_model.predict(x_test_rf)

y_pred_rf_actual   = np.expm1(y_pred_rf_log)
y_test_rf_actual   = np.expm1(y_test_rf)


mae_rf  = mean_absolute_error(y_test_rf_actual, y_pred_rf_actual)
rmse_rf = np.sqrt(mean_squared_error(y_test_rf_actual, y_pred_rf_actual))
r2_rf   = r2_score(y_test_rf_actual, y_pred_rf_actual)

# 6. Print Hasil
print(f"\n---- Random Forest Results ----")
print(f"MAE  : BDT {mae_rf:,.0f}")
print(f"RMSE : BDT {rmse_rf:,.0f}")
print(f"R2   : {r2_rf:.4f} ({r2_rf*100:.2f}%)")

In [ ]:
contribution = rf_model.feature_importances_

feature_names = x_train_rf.columns

df_importance = pd.DataFrame({
    'Feature': feature_names,
    'Contribution (%)': contribution * 100
}).sort_values(by='Contribution (%)', ascending=False).reset_index(drop= True)

print("------Random Forest Contribution Feature------")
#df_importance 

In [ ]:
plt.figure(figsize=(9, 5))

ax = sns.barplot(
    data=df_importance.head(8), 
    x='Contribution (%)', 
    y='Feature', 
    palette='Blues_r'
)
    
for p in ax.patches:
    width = p.get_width()
    ax.annotate(f'{width:.1f}%',
                (width + 0.5, p.get_y() + p.get_height() / 2.),
                ha='left', 
                va='center', 
                fontsize=10, 
                fontweight='bold',
                color='black')
    
plt.title('Top 10 Price Drivers Property', fontweight='bold')
plt.xlabel('Importance')
plt.xlim(0, max(df_importance['Contribution (%)']) * 1.15)
plt.tight_layout()
plt.show()